In [1]:
import time
from typing import Dict, Any

In [8]:
class AHE:
    def __init__(self, q: int, m: int, n: int, w:int):
        self.q = q
        self.m = m
        self.n = n
        self.w = w
        
        self.F = GF(q)
        self.Fext = GF(q^m, 'z')
        
        Tmp.<y> = PolynomialRing(self.F)
        Q = Tmp.random_element(degree=self.n)
        while not Q.is_irreducible():
            Q = Tmp.random_element(degree=self.n)
        
        self.PR = PolynomialRing(self.Fext, 'x')
        self.x = self.PR.gen()
        self.Q = self.PR(Q)

    @staticmethod
    def vec(a):
        return vector(list(a))

    def _mat(self, a):
        return matrix(self.F, [list(elem) for elem in a]).T

    def _rank_norm(self, a):
        return rank(self._mat(a))

    def _find_B(self, f):
        B = self._mat(f)
        cand_cols = identity_matrix(self.F, self.m)
        for i in range(self.m):
            tmp = B.augment(cand_cols[:,i])
            if rank(tmp) == rank(B) + 1:
                B = tmp
            if B.ncols() == self.m:
                break
    
        return B

    def _sample_with_supp(self, f):
        c = random_vector(self.F, len(f))
    
        return f.inner_product(c)

    def _poly(self, v):
        return self.PR(list(v))

    def _poly_to_vec(self, p, d):
        p_list = list(p)
        
        return vector(self.Fext, p_list + (d+1-len(p_list))*[0])

    def _vec_vec_prod(self, u, v):
        u_poly = self._poly(u)
        v_poly = self._poly(v)
        tmp = (u_poly * v_poly) % self.Q
    
        return self._poly_to_vec(tmp, len(u)-1)

    def _get_f(self): 
        f = random_vector(self.Fext, self.w)
        while self._rank_norm(f) != self.w:
            f = random_vector(self.Fext,self.w)
    
        return f

    def _get_g(self, B):
        z = self.Fext.gen() 
    
        z_pows = vector(self.Fext,[z**i for i in range(self.m)])
        g = z_pows * B[:,self.w:]

        return g

    def key_gen(self):
        f = self._get_f()
        B = self._find_B(f)
        g = self._get_g(B)
        D = (B**(-1)).T[:,self.w:]
        s = vector(self.Fext, [self._sample_with_supp(f) for _ in range(self.n)])

        return f,g,D,s

    def encrypt(self, message, key):
        f,g,D,s = key
        
        u = random_vector(self.Fext, self.n)
        p = self._poly(u)
        R2 = random_matrix(self.F,self.w,self.n)
        e = f*R2
        v = self._vec_vec_prod(s,u) + e + (message*g[0])
        ct = (u,v)
    
        return ct

    def decrypt(self, ct, key):
        f,g,D,s = key
        
        u,v = ct
        d = D.column(0)

        tmp = v - self._vec_vec_prod(s, u)
    
        message = [d.inner_product(self.vec(tmp[j])) for j in range(len(tmp))]
    
        return vector(message)


    @staticmethod
    def sum(ct1, ct2):
        return (ct1[0] + ct2[0], ct1[1] + ct2[1])

    def mul_const(self, ct, msg):
        u = self._vec_vec_prod(ct[0], msg)
        v = self._vec_vec_prod(ct[1], msg)

        return (u, v)

In [9]:
ahe = AHE(2, 15, 11, 3)
sk = ahe.key_gen()

msg1 = random_vector(ahe.F, ahe.n)
msg2 = random_vector(ahe.F, ahe.n)

ct1 = ahe.encrypt(msg1, sk)
ct2 = ahe.encrypt(msg2, sk)

ct3 = ahe.sum(ct1,ct2)

print(ahe.decrypt(ct3,sk) == msg1+msg2)

True


In [10]:
class SHE(AHE):
    def _get_g(self, B, d_F):
        z = self.Fext.gen() 
    
        z_pows = vector(self.Fext,[z**i for i in range(self.m)])
        g = z_pows * B[:,d_F:]
    
        return g

    def key_gen(self):
        f = self._get_f()
        
        rw = 0
        d_F = 1
        while d_F + 2!= rw:
            g1 = self.Fext.random_element()
            fg1 = f*g1
        
            gens = (
                list(f) +
                list(fg1) +
                [f[i] * f[j] for i in range(self.w) for j in range(i, self.w)]
            )
            
            M = matrix(self.F, [self.vec(g) for g in gens])
        
            F_ = M.row_space().basis()
            g2 = g1*g1
        
            f_ = vector(self.Fext, [self.Fext(list(b)) for b in F_])
            check = vector(self.Fext, list(f_) + [g1, g2])
    
            d_F = len(f_)
            rw = self._rank_norm(check)
    
        B = self._find_B(check)
        g = self._get_g(B,d_F)
        D = (B**(-1)).T[:,d_F:]
        s = vector(self.Fext, [self._sample_with_supp(f) for _ in range(self.n)])
    
        return f,g,D,s

    def mul_ct(self, ct1, ct2):
        u1,v1 = ct1
        u2,v2 = ct2
    
        a = self._vec_vec_prod(v1,v2)
        b = -(self._vec_vec_prod(u1,v2) + self._vec_vec_prod(u2,v1))
        c = self._vec_vec_prod(u1,u2)
    
        return a,b,c

    def decrypt_mul(self, ct, key):
        f,g,D,s = key
        a,b,c = ct
    
        sb = self._vec_vec_prod(s,b)
        ss = self._vec_vec_prod(s,s)
        s2c = self._vec_vec_prod(ss,c)
    
        tmp = a + sb + s2c
    
        d = D[:,1]
    
        message = (d).T * self._mat(tmp)
    
        return vector(message)

In [11]:
she = SHE(2, 15, 11, 3)
sk = she.key_gen()

m1 = random_vector(she.F, she.n)
m2 = random_vector(she.F, she.n)

ct1 = she.encrypt(m1, sk)
ct2 = she.encrypt(m2, sk)

ct_mul = she.mul_ct(ct1, ct2)
result = she.decrypt_mul(ct_mul, sk)
expected = she._vec_vec_prod(m1, m2)

print(result == expected)

True


In [12]:
params = [
    (2, 172, 20, 13),
    (2, 367, 183, 7),
    (2, 1296, 314, 6),
    (2, 3125, 713, 5)
]

In [13]:
for i in range(len(params)):
    print(f'Params: q={params[i][0]} m={params[i][1]} n={params[i][2]} w={params[i][3]}')
    she = SHE(*params[i])
    start = time.time()
    sk = she.key_gen()
    end = time.time()
    print(f'key gen time: {end - start}')

    msg1 = random_vector(she.F, she.n)
    msg2 = random_vector(she.F, she.n)

    start = time.time()
    ct1 = she.encrypt(msg1, sk)
    end = time.time()
    ct2 = she.encrypt(msg2, sk)

    print(f'enc time: {end-start}')
    
    start = time.time()
    res = she.sum(ct1, ct2)
    end = time.time()

    print(f'time sum: {end-start}')

    start = time.time()
    decrypt_sum = she.decrypt(res, sk)
    end = time.time()

    print(f'res sum: {decrypt_sum == msg1+msg2}\ndecrypt sum time: {end-start}')    

    start = time.time()
    res = she.mul_ct(ct1, ct2)
    end = time.time()

    print(f'time mul: {end-start}')

    start = time.time()
    decrypt_mul = she.decrypt_mul(res, sk)
    end = time.time()

    print(f'res mul: {decrypt_mul == she._vec_vec_prod(msg1, msg2)}\ndecrypt mul time: {end - start}\n===============\n')

Params: q=2 m=172 n=20 w=13
key gen time: 0.2461707592010498
enc time: 0.035920143127441406
time sum: 1.2159347534179688e-05
res sum: True
decrypt sum time: 0.031106948852539062
time mul: 0.08488297462463379
res mul: True
decrypt mul time: 0.06740713119506836

Params: q=2 m=367 n=183 w=7
key gen time: 0.32741713523864746
enc time: 0.736321210861206
time sum: 5.078315734863281e-05
res sum: True
decrypt sum time: 0.6231410503387451
time mul: 1.8148529529571533
res mul: True
decrypt mul time: 1.5052480697631836

Params: q=2 m=1296 n=314 w=6
key gen time: 3.640415906906128
enc time: 4.685543060302734
time sum: 0.0001380443572998047
res sum: True
decrypt sum time: 3.896761894226074
time mul: 11.472021102905273
res mul: True
decrypt mul time: 8.955106973648071

Params: q=2 m=3125 n=713 w=5
key gen time: 24.433109998703003
enc time: 26.928658962249756
time sum: 0.0009150505065917969
res sum: True
decrypt sum time: 23.86051034927368
time mul: 67.55226016044617
res mul: True
decrypt mul time: 5

In [56]:
class ObviousPIRClient:
    def __init__(self, she, sk):
        self.she = she
        self.sk = sk

    def query(self, i, N):
        vectors = [zero_vector(self.she.F, self.she.n) for _ in range(N)]
        vectors[i][0] = 1

        ct_vectors = []
        for v in vectors:
            ct = self.she.encrypt(v, self.sk)
            ct_vectors.append(ct)

        return ct_vectors

    def decode(self, ct):
        return self.she.decrypt(ct, self.sk)

In [65]:
class ObviousPIRServer:
    def __init__(self, she):
        self.she = she

    def answer(self, q, db):
        res = None
        for i in range(db.nrows()):
            if res is None:
                res = she.mul_const(q[i],db[i])
            else:
                res = she.sum(res, she.mul_const(q[i],db[i]))

        return res

In [72]:
db_len = 100
idx = 3
db = random_matrix(she.F,db_len,she.n)
print(db[idx])

client = ObviousPIRClient(she, sk)
q = client.query(idx, db_len)

server = ObviousPIRServer(she)
resp = server.answer(q, db)

res = client.decode(resp)
print(res == db[idx])

(1, 1, 1, 1, 0, 1, 0, 1, 0, 1, 1, 0, 1, 0, 0, 1, 1, 1, 0, 1)
True


In [ ]:
 class PIRClient:
    def __init__(self, she, sk):
         self.she = she
         self.sk = sk

    def query(self, i, N):
        block_cnt = ceil(N / self.she.n)
        vectors = [zero_vector(self.she.F, self.she.n) for _ in range(block_cnt)]
        
        block = i // self.she.n
        idx = i % self.she.n

        vectors[block][idx] = 1

        ct_vectors = []
        for v in vectors:
            ct = self.she.encrypt(v, self.sk)
            ct_vectors.append(ct)

        return ct_vectors

    def decode(self, ct, pos):
        return self.she.decrypt(ct, self.sk)[pos]

In [21]:
class PIRServer:
    def __init__(self, she):
        self.she = she

    def answer(self, q, db):
        res = None
        n = ceil(len(db) // self.she.n)
        for i in range(n):
            if res is None:
                res = she.mul_const(q[i],db[self.she.n*i:self.she.n+self.she.n*i])
            else:
                res += she.mul_const(q[i],db[self.she.n*i:self.she.n+self.she.n*i])

        return res

In [22]:
she = SHE(*params[0])
sk = she.key_gen()

In [23]:
db_len = 95
idx = 54
db = random_vector(she.F,db_len)
print(db[idx])

client = PIRClient(she, sk)

q = client.query(idx, db_len)

server = PIRServer(she)
resp = server.answer(q,db)

client.decode(resp, idx % she.n)

0


ValueError: too many values to unpack (expected 2)

In [37]:
a = Fext(1)
print(len(a.polynomial().coefficients(sparse=False)))
print(len(list(a)))

1
8


In [62]:
p = she.PR([she.Fext.random_element(), she.Fext(0), she.Fext(0)])
print(len(list(p)))
print(p.degree())
print([0]*3)

1
0
[0, 0, 0]
